<a href="https://colab.research.google.com/github/Pri-codes-10/leviathan_final/blob/main/leviathan_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


import numpy as np
import pandas as pd
import copy

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader



SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)



print("Loading data...")
train = pd.read_csv("train.csv")

sensor_cols = [c for c in train.columns if c.startswith("sensor_")]

X, y = [], []
for seq_id, group in train.groupby("sequence_id"):
    group = group.sort_values("timestep")
    X.append(group[sensor_cols].values)
    y.append(int(group["level"].iloc[0]))

X = np.array(X)
y = np.array(y)

print(f"X shape: {X.shape}") # Expected: (N, 10, 25)
print(f"y shape: {y.shape}") # Expected: (N,)



scaler = StandardScaler()
X_flat = X.reshape(-1, 25)
X_flat = scaler.fit_transform(X_flat)
X = X_flat.reshape(X.shape)



class LeviathanDataset(Dataset):
    def __init__(self, X, y=None, is_train=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None
        self.is_train = is_train

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x_val = self.X[idx]

        # Strategic Augmentation: Inject slight noise during training to prevent overfitting
        if self.is_train:
            noise = torch.randn_like(x_val) * 0.01
            x_val = x_val + noise

        if self.y is None:
            return x_val

        return x_val, self.y[idx]


class InvertedTransformer(nn.Module):
    def __init__(self, seq_len=10, num_sensors=25, d_model=256, nhead=8, num_layers=6, num_classes=4):
        super().__init__()

        # 1. Project the raw 10-timestep sequence of EACH sensor into a hidden dimension
        self.feature_projector = nn.Linear(seq_len, d_model)

        # 2. Sensor Embeddings: Gives the model "spatial" awareness of which sensor is which
        self.sensor_embeddings = nn.Parameter(torch.randn(1, num_sensors, d_model))

        # 3. CLS Token: A learnable token to aggregate all sensor information
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

        # 4. Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 2,
            batch_first=True,
            dropout=0.15,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 5. Classification Head
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # Initial shape: (Batch, 10, 25)

        # INVERT: Swap the time and sensor dimensions
        x = x.transpose(1, 2) # New shape: (Batch, 25, 10)

        # Project each sensor's 10-step history into d_model
        x = self.feature_projector(x) # New shape: (Batch, 25, d_model)

        # Add spatial/sensor embeddings
        x = x + self.sensor_embeddings

        # Prepend the CLS token
        B = x.size(0)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1) # Shape: (Batch, 26, d_model)

        # Pass through the Transformer
        x = self.transformer(x)

        # Extract the processed CLS token (at index 0)
        cls_out = x[:, 0, :]

        # Classify
        return self.mlp_head(cls_out)



class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    return total_loss / len(loader), correct / total

def validate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            preds.extend(outputs.argmax(1).cpu().numpy())
            labels.extend(y_batch.numpy())

    return accuracy_score(labels, preds)



skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []
best_models = []
EPOCHS = 60

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n========== FOLD {fold+1} ==========")

    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]

    # Notice `is_train=True` enables the Gaussian noise augmentation
    train_dataset = LeviathanDataset(X_train, y_train, is_train=True)
    val_dataset = LeviathanDataset(X_val, y_val, is_train=False)

    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

    # Class weights for Focal Loss
    weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
    weights = torch.tensor(weights, dtype=torch.float32).to(device)

    # Initialize Model, Loss, Optimizer
    model = InvertedTransformer(
        d_model=192,
        nhead=8,
        num_layers=4
    ).to(device)
    criterion = FocalLoss(alpha=weights, gamma=2.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)

    # Linear Warmup for the first 5 epochs, followed by Cosine Annealing
    warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=5)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS-5)
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[5])

    best_acc = 0
    patience = 15
    counter = 0
    best_state = None

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
        val_acc = validate(model, val_loader)

        # Step the unified scheduler
        scheduler.step()

        print(f"Fold {fold+1} Epoch {epoch+1:02d} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Loss: {train_loss:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            print(f"Early Stopping Triggered on Fold {fold+1}!")
            break

    model.load_state_dict(best_state)
    best_models.append(model)
    fold_scores.append(best_acc)
    print(f"Best Fold Accuracy: {best_acc:.4f}")


print("FINAL RESULTS")
for i, score in enumerate(fold_scores):
    print(f"Fold {i+1}: {score:.4f}")
print(f"\nMean CV Accuracy: {np.mean(fold_scores):.4f}")
print(f"Standard Deviation: {np.std(fold_scores):.4f}")

Loading data...
X shape: (30000, 10, 25)
y shape: (30000,)
Using device: cuda

========== FOLD 1 ==========
Fold 1 Epoch 01 | Train Acc: 0.2210 | Val Acc: 0.2248 | Loss: 0.7923
Fold 1 Epoch 02 | Train Acc: 0.2138 | Val Acc: 0.2012 | Loss: 0.7885
Fold 1 Epoch 03 | Train Acc: 0.2252 | Val Acc: 0.2012 | Loss: 0.7872
Fold 1 Epoch 04 | Train Acc: 0.2509 | Val Acc: 0.3037 | Loss: 0.7769
Fold 1 Epoch 05 | Train Acc: 0.4248 | Val Acc: 0.4468 | Loss: 0.6306
Fold 1 Epoch 06 | Train Acc: 0.5697 | Val Acc: 0.6548 | Loss: 0.4408
Fold 1 Epoch 07 | Train Acc: 0.6523 | Val Acc: 0.7445 | Loss: 0.3314
Fold 1 Epoch 08 | Train Acc: 0.7078 | Val Acc: 0.7145 | Loss: 0.2591
Fold 1 Epoch 09 | Train Acc: 0.7421 | Val Acc: 0.8028 | Loss: 0.2131
Fold 1 Epoch 10 | Train Acc: 0.7303 | Val Acc: 0.8035 | Loss: 0.2227
Fold 1 Epoch 11 | Train Acc: 0.7451 | Val Acc: 0.7828 | Loss: 0.1965
Fold 1 Epoch 12 | Train Acc: 0.7608 | Val Acc: 0.7543 | Loss: 0.1801
Fold 1 Epoch 13 | Train Acc: 0.7801 | Val Acc: 0.8347 | Loss: 0.

In [ ]:
# ==========================================
# LOAD TEST DATA
# ==========================================

test = pd.read_csv("test.csv")

sensor_cols = [c for c in test.columns if c.startswith("sensor_")]

X_test = []

for seq_id, group in test.groupby("sequence_id"):
    group = group.sort_values("timestep")
    X_test.append(group[sensor_cols].values)

X_test = np.array(X_test)

print("Test shape:", X_test.shape)

Test shape: (10000, 10, 25)


In [ ]:
# ==========================================
# APPLY TRAINING SCALER
# ==========================================

X_test_flat = X_test.reshape(-1, 25)

X_test_flat = scaler.transform(X_test_flat)

X_test = X_test_flat.reshape(X_test.shape)

In [ ]:
# ==========================================
# TEST DATASET
# ==========================================

test_dataset = LeviathanDataset(
    X_test,
    y=None,
    is_train=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=512,
    shuffle=False
)

In [ ]:
# ==========================================
# ENSEMBLE PREDICTIONS
# ==========================================

all_probs = []

for model in best_models:

    model.eval()

    fold_probs = []

    with torch.no_grad():

        for X_batch in test_loader:

            X_batch = X_batch.to(device)

            outputs = model(X_batch)

            probs = torch.softmax(outputs, dim=1)

            fold_probs.append(
                probs.cpu().numpy()
            )

    fold_probs = np.vstack(fold_probs)

    all_probs.append(fold_probs)

all_probs = np.mean(all_probs, axis=0)

preds = np.argmax(all_probs, axis=1)

print(preds.shape)

(10000,)


In [ ]:
# ==========================================
# CREATE SUBMISSION FILE
# ==========================================

submission = pd.DataFrame({
    "sequence_id": test["sequence_id"].unique(),
    "level": preds
})

submission.to_csv(
    "submission.csv",
    index=False
)

print(submission.head())
print("Submission saved!")

   sequence_id  level
0        30000      2
1        30001      3
2        30002      3
3        30003      3
4        30004      3
Submission saved!
